# 01 · The search object and its settings

The search object holds every setting of a run. Settings are grouped by purpose:

| group | controls |
|---|---|
| domain | the grid and the boundary excluded from fitting |
| preprocessing | how derivatives are computed and up to which order |
| search space | which terms may appear and how large an equation may be |
| objectives | how candidate equations are scored and sparsified |
| solver | optional checks of an equation by solving it |
| evolution | population size, number of epochs and evolutionary operators |
| runtime | memory for cached terms, logging |

Each setting has a default. A configuration (dictionary, JSON or YAML) overrides
the defaults, and keyword arguments override the configuration.

Displayed outputs are saved examples from earlier executions and do not verify the current version. Re-execute the relevant cells to obtain current results.

In [1]:
import os, sys, time
from pathlib import Path
os.environ.setdefault('OPENBLAS_NUM_THREADS', '1')   # before numpy: EPDE is faster single-threaded
os.environ.setdefault('OMP_NUM_THREADS', '1')

# find projects/pic (this notebook lives in projects/pic/notebooks)
PIC = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'epde_bench').is_dir())
sys.path[:0] = [str(PIC.parent.parent), str(PIC)]
import epde

import numpy as np
import matplotlib.pyplot as plt
from epde_bench import datasets, metrics, load_config, resolve_for_problem, discover
from epde_bench.plotting import plot_problem, plot_front
from epde_bench.truthcheck import check, print_check
from epde_bench.nbcache import cached_run, print_run
import epde_bench.nbcache as notebook_cache
if os.environ.get('EPDE_BENCH_NOTEBOOK_CACHE'):
    notebook_cache.CACHE_DIR = Path(os.environ['EPDE_BENCH_NOTEBOOK_CACHE'])
from epde_bench.env import seed_everything
seed_everything(0)
print('Search cache:', notebook_cache.CACHE_DIR)
from epde_bench.reconstruction import plot_reconstruction
%matplotlib inline
print('EPDE imported; notebook environment ready')

Search cache: <repo>/projects/pic/results/notebook_execution_2026-10-09/search_cache
EPDE imported; notebook environment ready


## Defaults of every group

In [2]:
from dataclasses import asdict
from epde.interface.search_config import load_search_config
defaults = load_search_config(None)
for group in ['domain', 'preprocessing', 'search_space', 'objectives', 'solver', 'evolution', 'runtime']:
    values = asdict(getattr(defaults, group))
    values.pop('operators', None)
    print(f'{group}:', values, '\n')

domain: {'boundary_width': 5, 'time_axis': 0} 

preprocessing: {'default_preprocessor_type': 'poly', 'preprocessor_kwargs': {}, 'max_deriv_order': 1} 

search_space: {'data_fun_pow': 1, 'deriv_fun_pow': 1, 'equation_terms_max_number': 6, 'equation_factors_max_number': 1, 'rps_amplification_cap': 100.0, 'tokens': ()} 

objectives: {'multiobjective_mode': True, 'discrepancy_metric': 'wape', 'second_objective': 'instability', 'complexity_metric': 'factors', 'instability_metric': 'chi2', 'single_objective_metric': 'discrepancy', 'anchor_on_residual': False, 'sparsity_cls': <class 'epde.operators.common.sparsity.VWSRSparsity'>, 'sparsity_kwargs': {}} 

solver: {'use_solver': False, 'solver_backend': 'autograd', 'device': 'cpu', 'pinn_loss_mult': 0.0, 'error_metric': 'rmse', 'deepxde_config': {'net': [64, 64, 64, 64], 'activation': 'tanh', 'optimizer': 'adam', 'lr': 0.0001, 'num_domain': 4000, 'num_test': 500, 'epochs': 2000, 'train_frac': 0.8, 'lbfgs_maxiter': 2000, 'loss_weight_mode': 'gra

## Overriding settings

In [3]:
from epde import EpdeSearch
search = EpdeSearch(config={'evolution': {'population_size': 16}}, training_epochs=3)
print(search.config.evolution.population_size, search.config.evolution.training_epochs)

Set self._mem_for_cache as 15
setting builder with <epde.optimizers.builder.StrategyBuilder object at 0x120e8ae40>
setting builder with <epde.optimizers.builder.StrategyBuilder object at 0x120e8ae40>
16 3


## Settings stored in files

The scripts and notebooks of this project keep their settings in YAML and merge
them in layers; a later layer wins:

1. the shared protocol: one broad token pool and one set of evolutionary settings
   for every record, so the user does not tune the pool to the problem;
2. the file of the record: only what this problem cannot do without (a special
   token, a wider boundary, a larger population for coupled systems);
3. a method variant compared by the benchmark;
4. overrides given on the command line.

The settings passed to the search object are printed below for the shared
protocol and for the Lorenz system.

In [4]:
print((PIC / 'configs' / '_base.yaml').read_text())

# Common protocol for every data set.
#
# The idea (from the group's former benchmark): one
# broad token pool and one set of evolutionary settings for all problems, so
# the user does not tune the pool per problem. Data-set files only add what a
# problem cannot do without (a special token, a longer boundary, a bigger
# population for coupled systems).
#
# The search section is passed to the search object unchanged; every key has a
# documented default in the library.

loader: {}                       # keyword arguments of the data-set loader

search:
  domain:
    boundary_width: auto         # 10 % of each axis is dropped at the edges
  preprocessing:
    default_preprocessor_type: FD
    preprocessor_kwargs: {}
    max_deriv_order: auto        # 2 in time, 4 in every space axis
  search_space:
    data_fun_pow: 3              # u, u^2, u^3
    deriv_fun_pow: 2             # u_x, (u_x)^2
    equation_terms_max_number: 10
    equation_factors_max_number:
      factors_num: [1, 2]
  

In [5]:
print((PIC / 'configs' / 'lorenz.yaml').read_text())
import yaml
problem = datasets.load('lorenz')
print(yaml.safe_dump(resolve_for_problem(load_config('lorenz'), problem), sort_keys=False))

# Three coupled first-order equations on a window of the attractor chosen by
# the loader. Boundary as in the group's solver studies.
search:
  domain: {boundary_width: 10}
  preprocessing: {max_deriv_order: [1]}
  evolution: {population_size: 48}

domain:
  boundary_width: 10
preprocessing:
  default_preprocessor_type: FD
  preprocessor_kwargs: {}
  max_deriv_order:
  - 1
search_space:
  data_fun_pow: 3
  deriv_fun_pow: 2
  equation_terms_max_number: 10
  equation_factors_max_number:
    factors_num:
    - 1
    - 2
    probas:
    - 0.65
    - 0.35
  tokens:
  - family: grid
    labels:
    - x
    max_power: 2
    dimensionality: 0
  - family: fixed_trigonometric
    freq: 2.0
    dimensionality: 0
objectives: {}
solver:
  use_solver: false
  device: cpu
evolution:
  population_size: 48
  training_epochs: 5
runtime:
  verbose_params:
    show_iter_idx: false
    candidate_objectives: false
  memory_for_cache: 2



Values marked as automatic in the shared protocol are filled for each record: the
boundary is 10 % of every axis, derivatives go up to order 2 in time and 4 in
space, and the dimensionality of the token families follows the grid.

## Method variants

The benchmark compares variants that differ from the shared protocol in one
setting. Each entry is merged on top of the record's configuration.

In [6]:
print((PIC / 'configs' / 'variants.yaml').read_text())

# Method variants compared by the benchmark. Each entry is merged on top of the
# shared protocol and the file of the record. The default variant changes
# nothing: EPDE as shipped (percentage discrepancy, instability as the second
# Pareto axis, variance-weighted sparsity) with the finite differences of the
# shared protocol.

default: {}

# Old EPDE: L2 discrepancy, LASSO sparsity, complexity as the second axis
# (the earlier pipeline of the group's former benchmark).
legacy:
  search:
    objectives:
      discrepancy_metric: l2
      sparsity_cls: lasso
      sparsity_kwargs: {initial_sparsity_interval: [1.0e-12, 1.0e-4]}
      second_objective: complexity

# Derivatives from local polynomial fits instead of finite differences.
poly:
  search:
    preprocessing:
      default_preprocessor_type: poly

# Support chosen by the knee of the residual-vs-size curve.
knee:
  search:
    objectives:
      sparsity_cls: knee

# Instability measured by cross-validation over windows.
instab_cv